# DAY 7 - Data Cleaning: Real-World Messy Data

**80% of a data analyst's job is cleaning data.**

Real data is always dirty:
- Missing values
- Duplicates
- Wrong data types
- Inconsistent formats
- Typos and garbage values
- Outliers

---

## Topics Covered
1. Creating Realistic Messy Data
2. Finding Problems (Audit)
3. Handling Missing Values (5 strategies)
4. Removing Duplicates
5. Fixing Data Types
6. String Cleaning
7. Outlier Detection & Treatment
8. Data Standardization
9. Feature Engineering
10. Saving Clean Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
print('Libraries loaded!')

---
## 1. Create Realistic Messy Data

In [ ]:
# This is what real data looks like — messy!
np.random.seed(42)

raw_data = {
    'emp_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
               101, 112, 113, 114, 115],          # 101 is duplicate
    'name': ['Alice', 'bob', '  Charlie  ', 'DIANA', 'Eve',
              'frank', 'Grace', None, 'Henry', 'Iris',
              'Alice', 'Jake', 'karen', ' Leo ', 'Mia'],
    'age': [25, 30, 35, 28, -5, 150, 32, 27, 29, 31,
            25, 26, 33, 34, None],               # -5 and 150 = outliers
    'salary': [50000, 60000, None, 80000, 45000, 70000, None, 55000, 62000, 75000,
               50000, 48000, 66000, None, 72000],
    'department': ['IT', 'HR', 'it', 'Finance', 'HR', 'It', 'Finance', 'IT', 'HR', 'Finance',
                   'IT', 'hr', 'IT', 'Finance', 'HR'],  # inconsistent case
    'join_date': ['2020-01-15', '2019-03-22', '15/06/2021', '2018-11-30', '22-08-2022',
                  '2021-07-01', '2020/05/14', '2019-09-10', '2023-02-18', '2020-12-05',
                  '2020-01-15', '2022-04-25', '2019/11/08', '2021-03-17', '2020-06-30'],
    'email': ['alice@company.com', 'BOB@company.com', 'charlie@company.com', 'diana@company.COM',
              'eve@company.com', 'frank@company.com', 'grace@company.com', 'henry@company.com',
              'invalid-email', 'iris@company.com', 'alice@company.com', 'jake@company.com',
              'karen@company.com', 'leo@company.com', 'mia@company.com'],
    'phone': ['9876543210', '9123456789', 'N/A', '9876543210', '9234567890',
              '91-9345678901', '9456789012', '9567890123', '0000000000', '9678901234',
              '9876543210', '9789012345', '9890123456', None, '9012345678']
}

df_raw = pd.DataFrame(raw_data)
print('Raw Data:')
print(df_raw)
print(f'\nShape: {df_raw.shape}')

---
## 2. Step 1: Data Audit — Find All Problems

In [ ]:
print('='*60)
print('DATA AUDIT REPORT')
print('='*60)

print(f'\nShape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')

print('\nColumn Info:')
df_raw.info()

print('\nMissing Values:')
missing = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
print(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})[missing > 0])

print(f'\nDuplicate Rows: {df_raw.duplicated().sum()}')
print(f'Duplicate emp_id: {df_raw["emp_id"].duplicated().sum()}')

In [ ]:
print('Unique departments (before cleaning):')
print(df_raw['department'].unique())

print('\nAge range:', df_raw['age'].min(), 'to', df_raw['age'].max())
print('Salary stats:')
print(df_raw['salary'].describe())

---
## 3. Step 2: Handle Missing Values

5 Strategies:
1. Drop rows (when very few are missing)
2. Fill with mean/median (for numbers)
3. Fill with mode (for categories)
4. Fill with constant/placeholder
5. Forward/Backward fill (for time series)

In [ ]:
df = df_raw.copy()  # Always work on a copy!

# Strategy 1: Fill numeric with median (robust to outliers)
df['salary'] = df['salary'].fillna(df['salary'].median())
print('Salary missing filled with median:', df['salary'].median())

# Strategy 2: Fill numeric with mean
age_mean = df['age'].mean()
df['age_temp'] = df['age'].fillna(age_mean)
print(f'Age missing: fill with mean = {age_mean:.1f}')

# Strategy 3: Drop rows where name is missing (critical field)
before_drop = len(df)
df = df.dropna(subset=['name'])
print(f'Rows before dropping null names: {before_drop}')
print(f'Rows after dropping null names : {len(df)}')

print('\nRemaining missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

---
## 4. Step 3: Remove Duplicates

In [ ]:
print('Before removing duplicates:', len(df))
print('Duplicate rows:')
print(df[df.duplicated(keep=False)])

# Remove duplicate emp_id — keep first occurrence
df = df.drop_duplicates(subset=['emp_id'], keep='first')
print(f'\nAfter removing duplicates: {len(df)}')

---
## 5. Step 4: Fix Inconsistent Text Data

In [ ]:
# Clean NAME column
df['name'] = (
    df['name']
    .str.strip()        # Remove leading/trailing spaces
    .str.title()        # Title Case (Alice, Bob)
)
print('Names cleaned:')
print(df['name'].tolist())

# Clean DEPARTMENT column
df['department'] = (
    df['department']
    .str.strip()
    .str.upper()
    .replace({'IT': 'IT', 'HR': 'HR', 'FINANCE': 'Finance'})
)
df['department'] = df['department'].str.title()
print('\nDepartments after cleaning:')
print(df['department'].unique())

# Clean EMAIL column
df['email'] = df['email'].str.lower().str.strip()
df['email_valid'] = df['email'].str.contains(r'^[\w.-]+@[\w.-]+\.\w+$', regex=True)
print('\nInvalid emails:')
print(df[~df['email_valid']][['name', 'email']])

In [ ]:
# Clean PHONE column
def clean_phone(phone):
    if pd.isna(phone) or phone in ['N/A', '0000000000']:
        return None
    cleaned = str(phone).replace('-', '').replace(' ', '')
    if cleaned.startswith('91') and len(cleaned) == 12:
        cleaned = cleaned[2:]
    return cleaned if len(cleaned) == 10 else None

df['phone'] = df['phone'].apply(clean_phone)
print('Phone numbers cleaned:')
print(df[['name', 'phone']])

---
## 6. Step 5: Fix Data Types + Date Parsing

In [ ]:
# Convert join_date to proper datetime
# Multiple formats in the data — use infer_datetime_format
df['join_date'] = pd.to_datetime(df['join_date'], infer_datetime_format=True, errors='coerce')
print('Dates parsed:')
print(df[['name', 'join_date']])
print('\nData types after fixing:')
print(df[['join_date']].dtypes)

In [ ]:
# Extract useful date parts
df['join_year']  = df['join_date'].dt.year
df['join_month'] = df['join_date'].dt.month
df['tenure_years'] = (pd.Timestamp('today') - df['join_date']).dt.days // 365

print('Date features extracted:')
print(df[['name', 'join_date', 'join_year', 'join_month', 'tenure_years']])

---
## 7. Step 6: Outlier Detection & Treatment

In [ ]:
# First, fix age — it has obvious outliers (-5, 150)
print('Age values:', df['age'].tolist())

# Method 1: Hard limits (domain knowledge)
df['age_clean'] = df['age'].apply(lambda x: x if (pd.notna(x) and 18 <= x <= 65) else np.nan)
df['age_clean'] = df['age_clean'].fillna(df['age_clean'].median())
print('\nAge after fixing outliers:')
print(df['age_clean'].tolist())

In [ ]:
# Method 2: IQR method for salary
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f'Salary IQR Analysis:')
print(f'  Q1 = {Q1:,.0f}')
print(f'  Q3 = {Q3:,.0f}')
print(f'  IQR = {IQR:,.0f}')
print(f'  Lower Bound = {lower_bound:,.0f}')
print(f'  Upper Bound = {upper_bound:,.0f}')

outliers = df[(df['salary'] < lower_bound) | (df['salary'] > upper_bound)]
print(f'\nSalary Outliers ({len(outliers)} found):')
print(outliers[['name', 'salary']])

# Cap outliers (Winsorization)
df['salary'] = df['salary'].clip(lower=lower_bound, upper=upper_bound)
print('\nSalary after capping outliers:')
print(df['salary'].describe())

In [ ]:
# Visualize outliers before vs after
np.random.seed(42)
orig = np.concatenate([np.random.normal(60000, 10000, 50), [200000, -10000, 180000]])
capped = np.clip(orig, orig.mean() - 2*orig.std(), orig.mean() + 2*orig.std())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(orig)
axes[0].set_title('Before Outlier Treatment')
axes[0].set_ylabel('Salary (Rs)')
axes[1].boxplot(capped)
axes[1].set_title('After Outlier Treatment')
axes[1].set_ylabel('Salary (Rs)')
plt.tight_layout()
plt.show()

---
## 8. Step 7: Feature Engineering

Creating new columns from existing ones.

In [ ]:
# 1. Salary band
def salary_band(sal):
    if sal < 50000: return 'Junior'
    elif sal < 70000: return 'Mid'
    elif sal < 90000: return 'Senior'
    else: return 'Lead'

df['salary_band'] = df['salary'].apply(salary_band)

# 2. Age group
df['age_group'] = pd.cut(
    df['age_clean'],
    bins=[0, 25, 35, 45, 100],
    labels=['Young (<=25)', 'Mid (26-35)', 'Experienced (36-45)', 'Senior (45+)']
)

# 3. First name from full name
df['first_name'] = df['name'].str.split().str[0]

print('Feature Engineering Results:')
print(df[['name', 'salary', 'salary_band', 'age_clean', 'age_group', 'first_name']])

In [ ]:
# 4. One-Hot Encoding for Machine Learning
dept_encoded = pd.get_dummies(df['department'], prefix='dept')
print('Department encoded (for ML):')
print(dept_encoded.head())

# 5. Label encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['dept_encoded'] = le.fit_transform(df['department'])
print('\nDepartment Label Encoded:')
print(df[['department', 'dept_encoded']].drop_duplicates())

---
## 9. Step 8: Standardization & Normalization

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max Normalization (0 to 1)
scaler = MinMaxScaler()
df['salary_normalized'] = scaler.fit_transform(df[['salary']])

# Z-score Standardization (mean=0, std=1)
z_scaler = StandardScaler()
df['salary_standardized'] = z_scaler.fit_transform(df[['salary']])

print('Scaling Comparison:')
print(df[['salary', 'salary_normalized', 'salary_standardized']].round(3))

---
## 10. Final Clean Dataset

In [ ]:
# Build final clean dataframe
df_clean = df[[
    'emp_id', 'name', 'department', 'age_clean', 'salary',
    'salary_band', 'age_group', 'tenure_years', 'join_year', 'email'
]].rename(columns={'age_clean': 'age'})

print('FINAL CLEAN DATASET:')
print(df_clean)
print(f'\nShape: {df_clean.shape}')
print('\nMissing values:')
print(df_clean.isnull().sum())

# Save to CSV
df_clean.to_csv('employees_clean.csv', index=False)
print('\nSaved to employees_clean.csv')

---
## Data Cleaning Checklist

| Step | Action | Function |
|------|--------|----------|
| 1 | Check missing values | `df.isnull().sum()` |
| 2 | Check duplicates | `df.duplicated().sum()` |
| 3 | Check data types | `df.dtypes` |
| 4 | Fill missing numbers | `fillna(median/mean)` |
| 5 | Fill missing categories | `fillna(mode()[0])` |
| 6 | Drop duplicates | `drop_duplicates()` |
| 7 | Fix text case | `.str.strip().str.title()` |
| 8 | Parse dates | `pd.to_datetime()` |
| 9 | Fix outliers | IQR or domain knowledge |
| 10 | Create new features | Column operations |

---
## Homework

1. Create a messy dataset with 20 rows and 5 columns
2. Introduce missing values, duplicates, wrong types
3. Clean all issues step by step
4. Add 3 new feature-engineered columns
5. Save the clean version as CSV